# Notebook 31: Neural Candidate Pool Resolver

This notebook tests the immediate consequence of Notebook `30`'s live result: the hypothesis-forced branch run generated a small resolver candidate pool whose oracle contains the true diagnosis in every one of the 49 live cases, but the hand-built resolver selected only `44/49` correctly.

The control question is:

> Can a train/validate-derived neural candidate resolver choose better from Notebook `30`'s graph/Bayes/MLP/branch candidate pool without using 49-case labels for training or threshold selection?

The selected candidate policy is fixed before live-label evaluation:

```text
compact_neural_candidate_resolver_v1
features = graph + Bayes + MLP + branch/candidate-role + request-state features
model = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4)
selection = argmax neural candidate score within the candidate pool
```

The notebook also reports the candidate-pool oracle ceiling as a diagnostic upper bound. That oracle uses the evaluation label and is not a deployable method.

## 1. Utility Functions

In [ ]:
from __future__ import annotations

import ast
import json
import math
import random
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 13
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
NOTEBOOK30_ROOT = PROJECT_ROOT / "artifacts" / "trajectory_replicates" / "hypothesis_forced_differential_branching_49case_v1"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "trajectory_replicates" / "neural_candidate_pool_resolver_49case_v1"
FIGURE_DIR = ARTIFACT_ROOT / "figures"

RUN_NAME = "neural_candidate_pool_resolver_49case_v1"
SELECTED_POLICY_NAME = "compact_neural_candidate_resolver_v1"

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def as_python_list(value: Any) -> List[Any]:
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(value)
            except Exception:
                return []
    return []


def softmax(values: Iterable[float]) -> np.ndarray:
    arr = np.asarray(list(values), dtype=float)
    if arr.size == 0:
        return arr
    arr = arr - np.nanmax(arr)
    exp = np.exp(arr)
    denom = exp.sum()
    if denom <= 0 or not np.isfinite(denom):
        return np.ones_like(exp) / len(exp)
    return exp / denom


def topk_contains(df: pd.DataFrame, k: int, diff_col: str = "ranked_differential") -> Tuple[int, List[Dict[str, Any]]]:
    misses = []
    for _, row in df.iterrows():
        ranked = as_python_list(row.get(diff_col, []))
        if row["true_pathology"] not in ranked[:k]:
            misses.append({
                "case_id": row["case_id"],
                "true_pathology": row["true_pathology"],
                f"top{k}": ranked[:k],
            })
    return len(df) - len(misses), misses


def candidate_pool_oracle(candidate_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for case_id, group in candidate_df.groupby("case_id", sort=False):
        predicted = group["predicted_pathology"].dropna().astype(str).tolist()
        unique_predicted = sorted(set(predicted))
        true_pathology = group["true_pathology"].iloc[0]
        role_counts = group["candidate_role"].value_counts().to_dict()
        rows.append({
            "case_id": case_id,
            "true_pathology": true_pathology,
            "candidate_rows": int(len(group)),
            "unique_candidate_diagnoses": int(len(unique_predicted)),
            "base_candidate_rows": int(role_counts.get("base", 0)),
            "pseudo_candidate_rows": int(role_counts.get("pseudo", 0)),
            "branch_candidate_rows": int(role_counts.get("branch", 0)),
            "true_in_candidate_pool": bool((group["predicted_pathology"] == true_pathology).any()),
            "candidate_diagnoses": json.dumps(unique_predicted),
        })
    return pd.DataFrame(rows)


def select_by_score(candidate_df: pd.DataFrame, score_col: str, prefix: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    group_col = "case_id" if "case_id" in candidate_df.columns else "synthetic_state_id"
    scored = candidate_df.copy()
    scored[f"{prefix}_rank"] = scored.groupby(group_col)[score_col].rank(ascending=False, method="first").astype(int)
    scored[f"{prefix}_group_probability"] = scored.groupby(group_col)[score_col].transform(lambda s: softmax(s).tolist())
    scored[f"selected_by_{prefix}"] = scored[f"{prefix}_rank"] == 1

    selected = scored[scored[f"selected_by_{prefix}"]].copy()
    margins = []
    for case_id, group in scored.groupby(group_col, sort=False):
        values = group.sort_values(score_col, ascending=False)[score_col].astype(float).to_numpy()
        margin = float(values[0] - values[1]) if len(values) > 1 else float("nan")
        margins.append((case_id, margin))
    margin_df = pd.DataFrame(margins, columns=[group_col, f"{prefix}_score_margin"])
    selected = selected.merge(margin_df, on=group_col, how="left")
    return scored, selected


def classification_metrics(case_df: pd.DataFrame, correct_col: str = "correct") -> Dict[str, Any]:
    correct = case_df[correct_col].astype(bool)
    return {
        "num_cases": int(len(case_df)),
        "num_correct": int(correct.sum()),
        "accuracy": float(correct.mean()) if len(case_df) else float("nan"),
    }


def save_bar(path: Path, labels: List[str], values: List[float], ylabel: str, title: str, ylim: Tuple[float, float] | None = None) -> None:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(labels, values, color=["#4C78A8", "#59A14F", "#E15759", "#F28E2B", "#B07AA1"][: len(labels)])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(axis="y", alpha=0.25)
    for i, value in enumerate(values):
        label = f"{value:.3f}" if isinstance(value, float) and value <= 1 else f"{value:.0f}"
        ax.text(i, value + (0.01 if ylim and ylim[1] <= 1.1 else 0.2), label, ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_hist(path: Path, values: Iterable[float], bins: Iterable[float], xlabel: str, title: str) -> None:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(list(values), bins=list(bins), color="#4C78A8", edgecolor="white")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Cases")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

## 2. Load Notebook 30 Candidate Pool And Synthetic Resolver Features

In [ ]:
required_inputs = {
    "candidate_resolver_train_validate_features": NOTEBOOK30_ROOT / "candidate_resolver_train_validate_features.csv",
    "candidate_level_live_scores": NOTEBOOK30_ROOT / "candidate_level_live_scores.csv",
    "predictions": NOTEBOOK30_ROOT / "predictions.csv",
    "metrics": NOTEBOOK30_ROOT / "metrics.json",
}
missing = [name for name, path in required_inputs.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required Notebook 30 artifacts: {missing}")

train_validate_raw = pd.read_csv(required_inputs["candidate_resolver_train_validate_features"])
live_candidates_raw = pd.read_csv(required_inputs["candidate_level_live_scores"])
notebook30_predictions = pd.read_csv(required_inputs["predictions"])
notebook30_metrics = json.loads(required_inputs["metrics"].read_text(encoding="utf-8"))

print("Notebook 30 candidate train/validate rows:", train_validate_raw.shape)
print("Notebook 30 live candidate rows:", live_candidates_raw.shape)
print("Notebook 30 case rows:", notebook30_predictions.shape)
print("Notebook 30 selected accuracy:", notebook30_metrics.get("accuracy"))

## 3. Feature Engineering And Candidate-Pool Oracle

In [ ]:
BASE_FEATURES = [
    "candidate_order",
    "candidate_graph_score",
    "candidate_graph_posterior",
    "candidate_graph_rank",
    "candidate_graph_positive_support",
    "candidate_graph_contradiction",
    "candidate_bayes_log_score",
    "candidate_bayes_posterior",
    "candidate_bayes_rank",
    "candidate_mlp_posterior",
    "candidate_mlp_rank",
    "is_base_candidate",
    "is_branch_candidate",
    "is_pseudo_candidate",
    "request_count",
    "visible_root_count",
    "branch_trigger_probability",
    "pair_coverage",
    "pair_missing_utility",
]

LIVE_TO_TRAIN_COLUMNS = {
    "case_id": "case_id",
    "branch_id": "branch_id",
    "candidate_role": "candidate_role",
    "branch_role_name": "branch_role_name",
    "true_pathology": "true_pathology",
    "predicted_pathology": "candidate_pathology",
    "correct": "candidate_label",
    "resolver_candidate_order": "candidate_order",
    "resolver_candidate_graph_score": "candidate_graph_score",
    "resolver_candidate_graph_posterior": "candidate_graph_posterior",
    "resolver_candidate_graph_rank": "candidate_graph_rank",
    "resolver_candidate_graph_positive_support": "candidate_graph_positive_support",
    "resolver_candidate_graph_contradiction": "candidate_graph_contradiction",
    "resolver_candidate_bayes_log_score": "candidate_bayes_log_score",
    "resolver_candidate_bayes_posterior": "candidate_bayes_posterior",
    "resolver_candidate_bayes_rank": "candidate_bayes_rank",
    "resolver_candidate_mlp_posterior": "candidate_mlp_posterior",
    "resolver_candidate_mlp_rank": "candidate_mlp_rank",
    "resolver_is_base_candidate": "is_base_candidate",
    "resolver_is_branch_candidate": "is_branch_candidate",
    "resolver_is_pseudo_candidate": "is_pseudo_candidate",
    "resolver_request_count": "request_count",
    "resolver_visible_root_count": "visible_root_count",
    "resolver_branch_trigger_probability": "branch_trigger_probability",
    "resolver_pair_coverage": "pair_coverage",
    "resolver_pair_missing_utility": "pair_missing_utility",
    "resolver_score": "notebook30_resolver_score",
    "raw_bayes_judge_score": "raw_bayes_judge_score",
    "selected_by_judge": "notebook30_selected_candidate",
}


def make_live_training_schema(live_df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    for source_col, target_col in LIVE_TO_TRAIN_COLUMNS.items():
        if source_col in live_df.columns:
            out[target_col] = live_df[source_col]
    out["synthetic_state_id"] = out["case_id"]
    out["split"] = "test"
    out["candidate_source"] = out["candidate_role"]
    return out


def add_resolver_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in BASE_FEATURES:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out[BASE_FEATURES] = out[BASE_FEATURES].replace([np.inf, -np.inf], np.nan)
    for col in BASE_FEATURES:
        median = out[col].median()
        out[col] = out[col].fillna(0.0 if pd.isna(median) else median)

    eps = 1e-6
    for col in ["candidate_graph_posterior", "candidate_bayes_posterior", "candidate_mlp_posterior"]:
        clipped = np.clip(out[col].astype(float), eps, 1 - eps)
        out[f"{col}_logit"] = np.log(clipped / (1 - clipped))

    for col in ["candidate_graph_rank", "candidate_bayes_rank", "candidate_mlp_rank", "candidate_order"]:
        out[f"{col}_inv"] = 1.0 / (1.0 + out[col].clip(lower=0))
        out[f"{col}_neg"] = -out[col]

    group = out.groupby("synthetic_state_id")
    out["candidate_pool_size"] = group["candidate_pathology"].transform("count")

    group_context_cols = [
        "candidate_graph_score",
        "candidate_graph_posterior",
        "candidate_bayes_log_score",
        "candidate_bayes_posterior",
        "candidate_mlp_posterior",
        "candidate_graph_rank_neg",
        "candidate_bayes_rank_neg",
        "candidate_mlp_rank_neg",
        "candidate_order_neg",
    ]
    for col in group_context_cols:
        out[f"{col}_group_max"] = group[col].transform("max")
        out[f"{col}_minus_group_max"] = out[col] - out[f"{col}_group_max"]

    out["graph_minus_bayes_rank"] = out["candidate_graph_rank"] - out["candidate_bayes_rank"]
    out["graph_minus_mlp_rank"] = out["candidate_graph_rank"] - out["candidate_mlp_rank"]
    out["bayes_minus_mlp_rank"] = out["candidate_bayes_rank"] - out["candidate_mlp_rank"]
    out["graph_bayes_score_product"] = out["candidate_graph_score"] * out["candidate_bayes_posterior"]
    out["mlp_bayes_product"] = out["candidate_mlp_posterior"] * out["candidate_bayes_posterior"]
    return out

train_validate = add_resolver_features(train_validate_raw)
live_candidates = add_resolver_features(make_live_training_schema(live_candidates_raw))

EXCLUDED_FEATURE_COLUMNS = {
    "split",
    "synthetic_state_id",
    "case_id",
    "branch_id",
    "candidate_role",
    "branch_role_name",
    "true_pathology",
    "candidate_pathology",
    "candidate_label",
    "candidate_source",
    "notebook30_selected_candidate",
}
FEATURE_COLUMNS = [
    col for col in train_validate.columns
    if col not in EXCLUDED_FEATURE_COLUMNS and pd.api.types.is_numeric_dtype(train_validate[col])
]

for col in FEATURE_COLUMNS:
    if col not in live_candidates.columns:
        live_candidates[col] = 0.0

pool_oracle = candidate_pool_oracle(live_candidates_raw)
pool_oracle.to_csv(ARTIFACT_ROOT / "candidate_pool_oracle_summary.csv", index=False)

ranked_top3_count, ranked_top3_misses = topk_contains(notebook30_predictions, 3)
ranked_top5_count, ranked_top5_misses = topk_contains(notebook30_predictions, 5)

oracle_summary = {
    "num_cases": int(len(pool_oracle)),
    "mean_candidate_rows": float(pool_oracle["candidate_rows"].mean()),
    "mean_unique_candidate_diagnoses": float(pool_oracle["unique_candidate_diagnoses"].mean()),
    "min_unique_candidate_diagnoses": int(pool_oracle["unique_candidate_diagnoses"].min()),
    "max_unique_candidate_diagnoses": int(pool_oracle["unique_candidate_diagnoses"].max()),
    "candidate_pool_oracle_correct": int(pool_oracle["true_in_candidate_pool"].sum()),
    "candidate_pool_oracle_accuracy": float(pool_oracle["true_in_candidate_pool"].mean()),
    "ranked_differential_top3_correct": int(ranked_top3_count),
    "ranked_differential_top3_accuracy": float(ranked_top3_count / len(notebook30_predictions)),
    "ranked_differential_top3_misses": ranked_top3_misses,
    "ranked_differential_top5_correct": int(ranked_top5_count),
    "ranked_differential_top5_accuracy": float(ranked_top5_count / len(notebook30_predictions)),
    "ranked_differential_top5_misses": ranked_top5_misses,
}
write_json(ARTIFACT_ROOT / "candidate_pool_oracle_summary.json", oracle_summary)

print(json.dumps(oracle_summary, indent=2))
print("Feature count:", len(FEATURE_COLUMNS))

## 4. Train Neural Candidate Resolver

In [ ]:
train_rows = train_validate[train_validate["split"] == "train"].copy()
validate_rows = train_validate[train_validate["split"] == "validate"].copy()

scaler = StandardScaler()
X_train = scaler.fit_transform(train_rows[FEATURE_COLUMNS]).astype(np.float32)
y_train = train_rows["candidate_label"].astype(int).to_numpy()
X_validate = scaler.transform(validate_rows[FEATURE_COLUMNS]).astype(np.float32)
y_validate = validate_rows["candidate_label"].astype(int).to_numpy()
X_live = scaler.transform(live_candidates[FEATURE_COLUMNS]).astype(np.float32)

MODEL_SPECS = {
    "l2_logistic_candidate_scorer_diagnostic": LogisticRegression(
        max_iter=2000,
        C=1.0,
        solver="lbfgs",
        random_state=RANDOM_SEED,
    ),
    SELECTED_POLICY_NAME: MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=150,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12,
        random_state=RANDOM_SEED,
    ),
    "wide_neural_candidate_resolver_diagnostic": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=150,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12,
        random_state=RANDOM_SEED,
    ),
    "regularized_wide_neural_candidate_resolver_diagnostic": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=150,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12,
        random_state=RANDOM_SEED,
    ),
}

validation_summaries = []
model_outputs: Dict[str, Dict[str, Any]] = {}

validate_positive_pool_rate = float((validate_rows.groupby("synthetic_state_id")["candidate_label"].sum() > 0).mean())

for model_name, model in MODEL_SPECS.items():
    model.fit(X_train, y_train)
    validate_score = model.predict_proba(X_validate)[:, 1]
    live_score = model.predict_proba(X_live)[:, 1]

    validate_scored = validate_rows.copy()
    validate_scored["candidate_score"] = validate_score
    _, validate_selected = select_by_score(validate_scored, "candidate_score", "validation_model")

    live_scored = live_candidates.copy()
    live_scored["candidate_score"] = live_score
    _, live_selected = select_by_score(live_scored, "candidate_score", "live_model")

    try:
        row_auc = float(roc_auc_score(y_validate, validate_score))
        row_ap = float(average_precision_score(y_validate, validate_score))
    except ValueError:
        row_auc = float("nan")
        row_ap = float("nan")

    validate_has_positive = (validate_rows.groupby("synthetic_state_id")["candidate_label"].sum() > 0).to_dict()
    validate_selected = validate_selected.copy()
    validate_selected["candidate_pool_contains_truth"] = validate_selected["synthetic_state_id"].map(validate_has_positive).fillna(False)
    validate_conditional = validate_selected[validate_selected["candidate_pool_contains_truth"]]

    validation_summaries.append({
        "model_name": model_name,
        "selected_policy": model_name == SELECTED_POLICY_NAME,
        "train_rows": int(len(train_rows)),
        "validate_rows": int(len(validate_rows)),
        "validate_groups": int(validate_rows["synthetic_state_id"].nunique()),
        "validate_candidate_pool_recall": validate_positive_pool_rate,
        "validate_row_auc": row_auc,
        "validate_row_average_precision": row_ap,
        "validate_group_argmax_accuracy_all_groups": float(validate_selected["candidate_label"].mean()),
        "validate_group_argmax_accuracy_conditional_on_candidate_pool": float(validate_conditional["candidate_label"].mean()),
        "live_accuracy_diagnostic_only": float(live_selected["candidate_label"].mean()),
        "live_correct_diagnostic_only": int(live_selected["candidate_label"].sum()),
    })
    model_outputs[model_name] = {
        "model": model,
        "validate_scores": validate_score,
        "live_scores": live_score,
        "validate_selected": validate_selected,
        "live_selected": live_selected,
    }

validation_summary = pd.DataFrame(validation_summaries)
validation_summary.to_csv(ARTIFACT_ROOT / "neural_resolver_validation_summary.csv", index=False)
print(validation_summary.to_string(index=False))

## 5. Policy Variant Evaluation

In [ ]:
selected_live_scores = model_outputs[SELECTED_POLICY_NAME]["live_scores"]
live_candidate_scores = live_candidates.copy()
live_candidate_scores["neural_score"] = selected_live_scores
live_candidate_scores["notebook30_resolver_score"] = live_candidate_scores.get("notebook30_resolver_score", np.nan)

candidate_level_neural, selected_neural_candidates = select_by_score(live_candidate_scores, "neural_score", "neural")

candidate_level_neural = candidate_level_neural.merge(
    pool_oracle[["case_id", "candidate_rows", "unique_candidate_diagnoses", "true_in_candidate_pool"]],
    on="case_id",
    how="left",
)

case_level = selected_neural_candidates[[
    "case_id",
    "branch_id",
    "candidate_role",
    "branch_role_name",
    "true_pathology",
    "candidate_pathology",
    "candidate_label",
    "neural_score",
    "neural_score_margin",
    "neural_group_probability",
    "candidate_pool_size",
]].copy()
case_level = case_level.rename(columns={
    "candidate_pathology": "neural_predicted_pathology",
    "candidate_label": "neural_correct",
    "branch_id": "neural_selected_branch_id",
    "candidate_role": "neural_selected_candidate_role",
    "branch_role_name": "neural_selected_branch_role_name",
})

case_level = case_level.merge(
    notebook30_predictions[[
        "case_id",
        "base_predicted_pathology",
        "base_correct",
        "predicted_pathology",
        "correct",
        "selected_candidate_role",
        "selected_branch_id",
        "num_requests_base",
        "num_requests",
        "total_branch_requests",
        "branch_trigger_fired",
        "branches_spawned",
    ]].rename(columns={
        "predicted_pathology": "notebook30_predicted_pathology",
        "correct": "notebook30_correct",
        "selected_candidate_role": "notebook30_selected_candidate_role",
        "selected_branch_id": "notebook30_selected_branch_id",
    }),
    on="case_id",
    how="left",
)
case_level = case_level.merge(pool_oracle[["case_id", "candidate_rows", "unique_candidate_diagnoses", "true_in_candidate_pool"]], on="case_id", how="left")

case_level["changed_vs_notebook30"] = case_level["neural_predicted_pathology"] != case_level["notebook30_predicted_pathology"]
case_level["improvement_vs_notebook30"] = case_level["neural_correct"].astype(bool) & ~case_level["notebook30_correct"].astype(bool)
case_level["regression_vs_notebook30"] = ~case_level["neural_correct"].astype(bool) & case_level["notebook30_correct"].astype(bool)
case_level["improvement_vs_base"] = case_level["neural_correct"].astype(bool) & ~case_level["base_correct"].astype(bool)
case_level["regression_vs_base"] = ~case_level["neural_correct"].astype(bool) & case_level["base_correct"].astype(bool)

candidate_level_neural.to_csv(ARTIFACT_ROOT / "candidate_level_neural_scores.csv", index=False)
case_level.to_csv(ARTIFACT_ROOT / "case_level_neural_resolver_results.csv", index=False)
case_level.to_csv(ARTIFACT_ROOT / "paired_notebook30_vs_neural_resolver.csv", index=False)

reference_rows = [
    {
        "system": "notebook30_base_branch",
        "num_correct": int(notebook30_metrics["base_num_correct"]),
        "accuracy": float(notebook30_metrics["base_accuracy"]),
        "mean_requests": float(notebook30_metrics["mean_base_requests"]),
        "mean_total_branch_requests": float(notebook30_metrics["mean_base_requests"]),
    },
    {
        "system": "notebook30_hand_resolver",
        "num_correct": int(notebook30_metrics["num_correct"]),
        "accuracy": float(notebook30_metrics["accuracy"]),
        "mean_requests": float(notebook30_metrics["mean_selected_requests"]),
        "mean_total_branch_requests": float(notebook30_metrics["mean_total_branch_requests"]),
    },
    {
        "system": SELECTED_POLICY_NAME,
        "num_correct": int(case_level["neural_correct"].sum()),
        "accuracy": float(case_level["neural_correct"].mean()),
        "mean_requests": float(case_level["num_requests"].mean()),
        "mean_total_branch_requests": float(case_level["total_branch_requests"].mean()),
    },
    {
        "system": "candidate_pool_oracle_diagnostic_only",
        "num_correct": int(pool_oracle["true_in_candidate_pool"].sum()),
        "accuracy": float(pool_oracle["true_in_candidate_pool"].mean()),
        "mean_requests": float(case_level["num_requests"].mean()),
        "mean_total_branch_requests": float(case_level["total_branch_requests"].mean()),
    },
]
policy_summary = pd.DataFrame(reference_rows)
policy_summary.to_csv(ARTIFACT_ROOT / "neural_resolver_policy_summary.csv", index=False)

selected_metrics = classification_metrics(case_level, "neural_correct")
selected_metrics.update({
    "wins_vs_notebook30": int(case_level["improvement_vs_notebook30"].sum()),
    "regressions_vs_notebook30": int(case_level["regression_vs_notebook30"].sum()),
    "changed_predictions_vs_notebook30": int(case_level["changed_vs_notebook30"].sum()),
    "wins_vs_base": int(case_level["improvement_vs_base"].sum()),
    "regressions_vs_base": int(case_level["regression_vs_base"].sum()),
    "mean_candidate_rows": float(pool_oracle["candidate_rows"].mean()),
    "mean_unique_candidate_diagnoses": float(pool_oracle["unique_candidate_diagnoses"].mean()),
    "candidate_pool_oracle_correct": int(pool_oracle["true_in_candidate_pool"].sum()),
    "candidate_pool_oracle_accuracy": float(pool_oracle["true_in_candidate_pool"].mean()),
    "mean_selected_requests": float(case_level["num_requests"].mean()),
    "mean_total_branch_requests": float(case_level["total_branch_requests"].mean()),
})

print(policy_summary.to_string(index=False))
print(json.dumps(selected_metrics, indent=2))

## 6. Paired Error Analysis

In [ ]:
hard_case_ids = sorted(set(
    case_level.loc[
        case_level["changed_vs_notebook30"]
        | ~case_level["neural_correct"].astype(bool)
        | ~case_level["notebook30_correct"].astype(bool),
        "case_id",
    ]
))

hard_case_audits = []
for case_id in hard_case_ids:
    case_row = case_level[case_level["case_id"] == case_id].iloc[0].to_dict()
    candidates = candidate_level_neural[candidate_level_neural["case_id"] == case_id].sort_values("neural_score", ascending=False)
    candidate_rows = []
    for _, cand in candidates.iterrows():
        candidate_rows.append({
            "branch_id": cand.get("branch_id"),
            "candidate_role": cand.get("candidate_role"),
            "branch_role_name": cand.get("branch_role_name"),
            "candidate_pathology": cand.get("candidate_pathology"),
            "true_pathology": cand.get("true_pathology"),
            "correct": bool(cand.get("candidate_label")),
            "selected_by_neural": bool(cand.get("selected_by_neural")),
            "neural_rank": int(cand.get("neural_rank")),
            "neural_score": float(cand.get("neural_score")),
            "neural_group_probability": float(cand.get("neural_group_probability")),
            "notebook30_resolver_score": None if pd.isna(cand.get("notebook30_resolver_score")) else float(cand.get("notebook30_resolver_score")),
            "graph_score": float(cand.get("candidate_graph_score")),
            "bayes_posterior": float(cand.get("candidate_bayes_posterior")),
            "mlp_posterior": float(cand.get("candidate_mlp_posterior")),
            "graph_rank": float(cand.get("candidate_graph_rank")),
            "bayes_rank": float(cand.get("candidate_bayes_rank")),
            "mlp_rank": float(cand.get("candidate_mlp_rank")),
        })
    hard_case_audits.append({
        "case": case_row,
        "candidates_ranked_by_neural_score": candidate_rows,
    })

write_json(ARTIFACT_ROOT / "hard_case_neural_resolver_audits.json", hard_case_audits)

misses = case_level[~case_level["neural_correct"].astype(bool)][[
    "case_id",
    "true_pathology",
    "base_predicted_pathology",
    "notebook30_predicted_pathology",
    "neural_predicted_pathology",
    "neural_selected_candidate_role",
    "neural_score",
    "neural_score_margin",
    "candidate_rows",
    "unique_candidate_diagnoses",
]]

wins = case_level[case_level["improvement_vs_notebook30"]][[
    "case_id",
    "true_pathology",
    "notebook30_predicted_pathology",
    "neural_predicted_pathology",
    "neural_selected_candidate_role",
    "neural_selected_branch_id",
    "neural_score",
    "neural_score_margin",
]]

print("Wins vs Notebook 30:")
print(wins.to_string(index=False))
print("\nRemaining neural misses:")
print(misses.to_string(index=False))

## 7. Figures

In [ ]:
save_bar(
    FIGURE_DIR / "accuracy_comparison.png",
    labels=policy_summary["system"].tolist(),
    values=policy_summary["accuracy"].tolist(),
    ylabel="Accuracy",
    title="Notebook 30 Candidate-Pool Resolver Accuracy",
    ylim=(0.80, 1.02),
)

save_hist(
    FIGURE_DIR / "candidate_pool_size_distribution.png",
    values=pool_oracle["unique_candidate_diagnoses"],
    bins=range(2, 10),
    xlabel="Unique candidate diagnoses",
    title="Notebook 30 Resolver Candidate-Pool Size",
)

fig, ax = plt.subplots(figsize=(8, 4.8))
plot_df = validation_summary.sort_values("validate_group_argmax_accuracy_all_groups", ascending=False)
ax.barh(plot_df["model_name"], plot_df["validate_group_argmax_accuracy_all_groups"], color="#4C78A8")
ax.set_xlabel("Validate group argmax accuracy")
ax.set_title("Resolver Validation Accuracy By Model")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "validation_model_comparison.png", dpi=160)
plt.close(fig)

paired_counts = pd.Series({
    "wins": int(case_level["improvement_vs_notebook30"].sum()),
    "regressions": int(case_level["regression_vs_notebook30"].sum()),
    "unchanged_correct": int((case_level["neural_correct"].astype(bool) & case_level["notebook30_correct"].astype(bool) & ~case_level["changed_vs_notebook30"]).sum()),
    "unchanged_wrong": int((~case_level["neural_correct"].astype(bool) & ~case_level["notebook30_correct"].astype(bool) & ~case_level["changed_vs_notebook30"]).sum()),
    "changed_wrong_to_wrong": int((~case_level["neural_correct"].astype(bool) & ~case_level["notebook30_correct"].astype(bool) & case_level["changed_vs_notebook30"]).sum()),
})
save_bar(
    FIGURE_DIR / "paired_outcomes_vs_notebook30.png",
    labels=paired_counts.index.tolist(),
    values=paired_counts.astype(float).tolist(),
    ylabel="Cases",
    title="Paired Outcomes: Neural Resolver vs Notebook 30",
)
paired_counts.rename_axis("paired_outcome").reset_index(name="case_count").to_csv(ARTIFACT_ROOT / "paired_outcome_counts.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.8))
role_counts = case_level["neural_selected_candidate_role"].value_counts().sort_index()
ax.bar(role_counts.index, role_counts.values, color="#59A14F")
ax.set_ylabel("Selected cases")
ax.set_title("Neural Resolver Selection Source")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "selection_source_counts.png", dpi=160)
plt.close(fig)

score_plot = case_level.copy()
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.scatter(score_plot["neural_score_margin"], score_plot["neural_correct"].astype(int), alpha=0.75, color="#4C78A8")
ax.set_xlabel("Neural score margin")
ax.set_ylabel("Correct selected diagnosis")
ax.set_yticks([0, 1])
ax.set_title("Neural Resolver Margin By Correctness")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "score_margin_by_correctness.png", dpi=160)
plt.close(fig)

hard_score_rows = []
for case_id in hard_case_ids:
    subset = candidate_level_neural[candidate_level_neural["case_id"] == case_id].sort_values("neural_score", ascending=False).head(8)
    for _, row in subset.iterrows():
        hard_score_rows.append({
            "case_id": case_id,
            "candidate": row["candidate_pathology"],
            "score": row["neural_score"],
            "correct": bool(row["candidate_label"]),
            "selected": bool(row["selected_by_neural"]),
        })
hard_score_df = pd.DataFrame(hard_score_rows)
hard_score_df.to_csv(ARTIFACT_ROOT / "hard_case_neural_score_ladders.csv", index=False)

print("Figures written to", FIGURE_DIR)

## 8. Final Summary And Artifact Contract

In [ ]:
promotion_decision = (
    "offline_candidate_promoted_for_followup_confirmation"
    if selected_metrics["num_correct"] >= notebook30_metrics["num_correct"] + 2 and selected_metrics["regressions_vs_notebook30"] == 0
    else "diagnostic_only_keep_prior_confirmed_method"
)

resolved_run_config = {
    "run_name": RUN_NAME,
    "artifact_root": str(ARTIFACT_ROOT.resolve()),
    "notebook30_artifact_root": str(NOTEBOOK30_ROOT.resolve()),
    "selected_policy_name": SELECTED_POLICY_NAME,
    "random_seed": RANDOM_SEED,
    "offline_only": True,
    "live_api_calls": 0,
    "features_used": FEATURE_COLUMNS,
    "excluded_inputs": [
        "49-case labels for model fitting",
        "49-case labels for threshold selection",
        "disease-name one-hot features in selected model",
    ],
    "model_spec": {
        "family": "sklearn.neural_network.MLPClassifier",
        "hidden_layer_sizes": [64, 32],
        "activation": "relu",
        "alpha": 1e-4,
        "selection_rule": "argmax neural candidate score within Notebook 30 candidate pool",
    },
}
write_json(ARTIFACT_ROOT / "resolved_run_config.json", resolved_run_config)

selected_policy = {
    "policy_name": SELECTED_POLICY_NAME,
    "status": promotion_decision,
    "method": "Compact neural candidate scorer over Notebook 30 graph/Bayes/MLP/branch candidate pool.",
    "selected_before_live_label_evaluation": True,
    "offline_only": True,
    "no_live_api_calls": True,
    "training_data": {
        "source": str((NOTEBOOK30_ROOT / "candidate_resolver_train_validate_features.csv").resolve()),
        "train_rows": int(len(train_rows)),
        "validate_rows": int(len(validate_rows)),
        "train_groups": int(train_rows["synthetic_state_id"].nunique()),
        "validate_groups": int(validate_rows["synthetic_state_id"].nunique()),
    },
    "notebook30_reference": {
        "num_cases": int(notebook30_metrics["num_cases"]),
        "base_correct": int(notebook30_metrics["base_num_correct"]),
        "base_accuracy": float(notebook30_metrics["base_accuracy"]),
        "selected_correct": int(notebook30_metrics["num_correct"]),
        "selected_accuracy": float(notebook30_metrics["accuracy"]),
        "mean_selected_requests": float(notebook30_metrics["mean_selected_requests"]),
        "mean_total_branch_requests": float(notebook30_metrics["mean_total_branch_requests"]),
    },
    "candidate_pool_oracle_diagnostic_only": oracle_summary,
    "selected_policy_metrics": selected_metrics,
    "promotion_decision": promotion_decision,
    "important_caveat": "The 49/49 candidate-pool oracle is a label-using diagnostic ceiling, not an achieved deployable policy.",
}
write_json(ARTIFACT_ROOT / "selected_neural_resolver.json", selected_policy)

summary_metrics = pd.DataFrame([{**selected_metrics, "promotion_decision": promotion_decision, "policy_name": SELECTED_POLICY_NAME}])
summary_metrics.to_csv(ARTIFACT_ROOT / "summary_metrics.csv", index=False)

required_outputs = [
    "resolved_run_config.json",
    "candidate_pool_oracle_summary.csv",
    "candidate_pool_oracle_summary.json",
    "neural_resolver_validation_summary.csv",
    "candidate_level_neural_scores.csv",
    "case_level_neural_resolver_results.csv",
    "paired_notebook30_vs_neural_resolver.csv",
    "hard_case_neural_resolver_audits.json",
    "selected_neural_resolver.json",
    "summary_metrics.csv",
    "neural_resolver_policy_summary.csv",
]
missing_outputs = [name for name in required_outputs if not (ARTIFACT_ROOT / name).exists()]
if missing_outputs:
    raise RuntimeError(f"Missing expected artifacts: {missing_outputs}")

print(json.dumps(selected_policy, indent=2)[:5000])
print("Artifact contract passed:", ARTIFACT_ROOT)